# 20｜視覺與多模態Transformer

把影像切成 patches，再以正規化向量比較圖文相似度。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/20_視覺與多模態Transformer.md`，目前 37 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

Patch/視窗/金字塔、DETR 配對、對比/自蒸餾、latent queries 與門控。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 20-03 | 7、8、20、21、22 | 課堂小實驗 |
| 20-04 | 3、20、21 | 執行結果圖 |
| 20-05 | 4、7、8、9、10、11、12、13、14、17、37 | 1. Patch、PVT 與視窗注意力 |
| 20-06 | 5、6、11、15、16、20、21、22、23、37 | 2. DETR 配對、CLIP 損失與自蒸餾 |
| 20-07 | 18、19、24、25、26、27、28、29、30、31、32、33、34、35、36、37 | 3. Perceiver、Q-Former 與交錯圖文的介面 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 20-01｜投影片 20：2 頁
import importlib.util, subprocess, sys
_needed = {'scipy': 'scipy>=1.11,<2', 'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 20-02｜投影片 20：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

圖文向量是手工指定，不是 CLIP encoder 的輸出。類別名稱只是數值例的標記，不證明圖片真的含該物件。

In [ ]:
# 程式 20-03｜投影片 20：7、8、20、21、22 頁
image = np.arange(16).reshape(4,4)
patches = np.stack([image[i:i+2,j:j+2].ravel() for i in (0,2) for j in (0,2)])
image_emb = np.array([[.9,.1], [.2,.8]])
text_emb = np.array([[1.,0.], [0.,1.], [.7,.7]])
labels = ['貓', '街景', '戶外動物']
norm = lambda x: x / np.linalg.norm(x, axis=1, keepdims=True)
sim = norm(image_emb) @ norm(text_emb).T
print('patch matrix:\n', patches)
pd.DataFrame(sim, index=['image_A','image_B'], columns=labels).round(3)

## 執行結果圖

In [ ]:
# 程式 20-04｜投影片 20：3、20、21 頁
plt.imshow(sim, cmap='viridis', vmin=0, vmax=1)
plt.xticks(range(3), ['cat','street','outdoor animal']); plt.yticks(range(2), ['image_A','image_B'])
plt.colorbar(label='cosine similarity'); plt.title('Normalized image-text similarity')
for (i,j), value in np.ndenumerate(sim): plt.text(j, i, f'{value:.2f}', ha='center', va='center', color='white')
savefig('20_multimodal_demo')
report('multimodal', {'patches': patches.tolist(), 'similarity': sim.tolist()})

## 1. Patch、PVT 與視窗注意力

影像分塊後依空間次序形成序列；多尺度合併縮短 token 數。視窗限制 attention 配對數，shift 必須處理邊界，不能把 cyclic roll 的對邊誤當鄰居。

In [ ]:
# 程式 20-05｜投影片 20：4、7、8、9、10、11、12、13、14、17、37 頁
rng20=np.random.default_rng(SEED);im20=rng20.normal(size=(2,3,16,16));P20=4
patch20=im20.reshape(2,3,4,P20,4,P20).transpose(0,2,4,1,3,5).reshape(2,16,3*P20*P20)
projection20=rng20.normal(0,.1,(48,8));embedding20=patch20@projection20
kernel20=projection20.T.reshape(8,3,4,4)
# Conv stride=patch size 與攤平後 Linear 等價
conv20=np.empty((2,16,8))
for i in range(4):
    for j in range(4):conv20[:,i*4+j]=np.einsum('bcij,ocij->bo',im20[:,:,i*4:i*4+4,j*4:j*4+4],kernel20)
assert np.allclose(conv20,embedding20)
cls20=np.zeros((2,1,8));distill20=np.ones((2,1,8))*.01;tokens20=np.concatenate([cls20,distill20,embedding20],axis=1)
assert tokens20.shape==(2,18,8);assert (224//16)**2==196
pyramid20=embedding20.reshape(2,4,4,8).reshape(2,2,2,2,2,8).mean(axis=(2,4)).reshape(2,4,8)
assert pyramid20.shape==(2,4,8)
# 同窗允許關注；shifted 視窗用真實座標分組，不循環連接對邊
coords20=np.array([(i,j) for i in range(4) for j in range(4)])
window_id20=coords20//2;shift_id20=(coords20+1)//2
allowed20=np.all(window_id20[:,None]==window_id20[None,:],axis=-1)
shift_allowed20=np.all(shift_id20[:,None]==shift_id20[None,:],axis=-1)
fig,axes=plt.subplots(1,3,figsize=(11,3));axes[0].imshow(im20[0,0]);axes[0].set_title('4 x 4 patches')
for x20 in [3.5,7.5,11.5]:axes[0].axhline(x20,c='r');axes[0].axvline(x20,c='r')
axes[1].imshow(allowed20,cmap='binary');axes[1].set_title('Window attention mask');axes[2].imshow(shift_allowed20,cmap='binary');axes[2].set_title('Shifted windows, no wrap');savefig('c20_patches_windows')
print('Global/window score entries:',16**2,int(allowed20.sum()),'Pyramid tokens:',pyramid20.shape[1])

## 2. DETR 配對、CLIP 損失與自蒸餾

DETR 以整體一對一成本配對；CLIP 以同批圖文對角線為正例。DINO 式教師以 EMA 更新，示範中心化與不同溫度；未完成多視角視覺訓練，不能宣稱避免了表示崩塌。

In [ ]:
# 程式 20-06｜投影片 20：5、6、11、15、16、20、21、22、23、37 頁
from scipy.optimize import linear_sum_assignment
pred_boxes20=np.array([[.2,.2,.3,.3],[.7,.6,.2,.4],[.1,.8,.1,.1]])
gt_boxes20=np.array([[.72,.6,.2,.4],[.2,.22,.3,.3]])
class_prob20=np.array([[.9,.1],[.2,.8],[.5,.5]]);gt_class20=np.array([1,0])
cost20=np.abs(pred_boxes20[:,None]-gt_boxes20[None]).sum(-1)-np.log(class_prob20[:,gt_class20])
rows20,cols20=linear_sum_assignment(cost20);print('DETR toy query->target:',list(zip(rows20,cols20)),'unmatched = no-object')
def softmax20(z):
    p=np.exp(z-z.max(-1,keepdims=True));return p/p.sum(-1,keepdims=True)
paired_image20=norm(image_emb);paired_text20=norm(text_emb[:2]);logit20=paired_image20@paired_text20.T/.1
row_prob20=softmax20(logit20);col_prob20=softmax20(logit20.T)
clip_loss20=-.5*(np.log(np.diag(row_prob20)).mean()+np.log(np.diag(col_prob20)).mean())
assert np.allclose(row_prob20.sum(1),1)
print('Toy symmetric CLIP loss:',clip_loss20,'candidate-dependent probabilities:',softmax20(sim/.1))
student20=rng20.normal(size=(8,4));teacher20=student20.copy();student20+=.1*rng20.normal(size=student20.shape)
teacher20=.9*teacher20+.1*student20
views20=embedding20.mean(1);teacher_logits20=views20@teacher20;center20=teacher_logits20.mean(0,keepdims=True)
target20=softmax20((teacher_logits20-center20)/.07).copy();student_prob20=softmax20(views20@student20/.1)
print('Self-distillation CE:',-np.sum(target20*np.log(student_prob20+1e-12),axis=1).mean())
fig,axes=plt.subplots(1,2,figsize=(9,3.5));axes[0].imshow(cost20,cmap='viridis');axes[0].set(xlabel='target',ylabel='object query',title='Matching cost')
axes[1].imshow(row_prob20,vmin=0,vmax=1,cmap='Blues');axes[1].set(xlabel='paired text',ylabel='image',title='Contrastive probabilities');savefig('c20_matching_contrast')

## 3. Perceiver、Q-Former 與交錯圖文的介面

以共享的 cross-attention 核心驗算 latent bottleneck、輸出查詢和圖片 token 接語言模型的形狀；不同模型的完整區塊、損失和訓練程序仍不相同。

In [ ]:
# 程式 20-07｜投影片 20：18、19、24、25、26、27、28、29、30、31、32、33、34、35、36、37 頁
def cross20(q,k,v):return softmax20(q@k.swapaxes(-1,-2)/np.sqrt(q.shape[-1]))@v
visual20=rng20.normal(size=(2,16,8));queries20=rng20.normal(size=(2,4,8))
latents20=cross20(queries20,visual20,visual20)
output_queries20=rng20.normal(size=(2,6,8));perceiver_output20=cross20(output_queries20,latents20,latents20)
assert latents20.shape==(2,4,8) and perceiver_output20.shape==(2,6,8)
# Q-Former 概念：少量學習 query 讀固定影像特徵，再投影到語言模型寬度
language_prefix20=latents20@rng20.normal(size=(8,12));text20=rng20.normal(size=(2,5,12));interleaved20=np.concatenate([language_prefix20,text20],axis=1)
assert interleaved20.shape==(2,9,12)
# Flamingo gate 初始化為0時，新增視覺分支不改原文字表示
gate20=0.;cross_image20=cross20(text20,language_prefix20,language_prefix20)
assert np.allclose(text20+np.tanh(gate20)*cross_image20,text20)
print('Perceiver / Q-query / interleaved shapes:',perceiver_output20.shape,latents20.shape,interleaved20.shape)
# 文字對齊、融合、條件生成是不同輸出契約
contracts20=pd.DataFrame({'task':['alignment','fusion','caption generation','image generation'],
 'output':['similarity matrix','fused features','token sequence','pixel/latent tensor']})
print(contracts20)

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。